# AnimationStudio - Progressive Identity Locking on Google Colab (T4)

Runs the four-script **Progressive Locking Pipeline** for selected characters:

1. `generate_identity_lock.py` - multi-angle reference sheets (front, 3/4, profile, back)
2. `generate_face_lock.py` - expression library
3. `generate_body_lock.py` - pose library
4. `generate_wardrobe.py` - outfit/wardrobe library

These are the runs that produce the curated reference/expression/pose/outfit
sets that feed `train_lora.py build-dataset` (>= 20 approved assets per
character). Without them the LoRA training golden path is broken.

## Drive budget (free tier = 5 GB)

- **Drive holds exactly one file**: `catalog.db` (the shortlisted/approved
  asset state). Nothing else is written to Drive.
- **Models live on the Colab disk** (`/content/models/`), NOT Drive - a Flux
  model is ~17.25 GB (fp8 Flux dev) and cannot fit a free account. It is
  re-downloaded after a VM reset (`wget -c` resumes). If you upgrade Drive,
  set `CACHE_MODELS_IN_DRIVE = True` in Cell 1 to keep them cached.
- **Generated images are exported into the Colab checkout** and then either
  pushed to GitHub or downloaded manually (Cell 11). No image bytes touch
  Drive.

## Branch = model flavor

| Branch | Model | Notes |
| --- | --- | --- |
| `colab-gpu` | fp8 Flux dev bundle (`flux1-dev.safetensors`, ~17.25 GB) | Best quality on the T4. One-file `CheckpointLoaderSimple`. |

## Steps

1. Runtime -> Change runtime type -> T4 GPU (or better).
2. In Cell 1 set `REPO_URL` to your GitHub clone URL and pick `CHARACTERS`.
3. Runtime -> Run all.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (colab-gpu is the only supported branch).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master is deprecated/unused.
BRANCH = "colab-gpu"  #@param ["colab-gpu"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
UI_PORT = 8000  #@param {type:"integer"}

# The ONLY thing stored on Google Drive (free tier = 5 GB): the asset DB.
# Shortlisted/approved state survives session resets here.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}
DB = f"{DRIVE_ROOT}/catalog.db"

# The ~17.25 GB model cache. Free Drive cannot hold it -> keep on the Colab
# disk (re-downloaded after a VM reset). Set True only if you have space.
CACHE_MODELS_IN_DRIVE = False  #@param {type:"boolean"}

# ---- Which characters get locked (Cell 3b previews before you burn GPU) ----
# Pick ONE character, or 'all' for the full 39-character universe.  Free-tier
# T4 note: each lock is ~2 min/image, and the identity lock alone is 80
# candidates x 4 angles. Prefer one character at a time on the free tier.
CHARACTERS = "Lily Bunny"  #@param ["all", "Ben Bear", "Charlie Fox", "Daisy Duck", "Lily Bunny", "Baby Bunny", "Daddy Bunny", "Grandma Bunny", "Grandpa Bunny", "Mommy Bunny", "Cat", "Chicken", "Cow", "Dog", "Elephant", "Horse", "Monkey", "Mouse", "Pig", "Sheep", "Chef Pig", "Construction Worker Beaver", "Doctor Panda", "Farmer Goat", "Firefighter Dalmatian", "Librarian Hedgehog", "Mail Carrier Turtle", "Musician Parrot", "Police Officer Beaver", "Teacher Owl", "Alien", "Cloud", "Friendly Dinosaur (Brontosaurus)", "Friendly Dragon", "Moon", "Rainbow", "Robot", "Stars", "Sun", "Unicorn"]

# Number of characters to process per lock stage before auto-stopping.
# 0 = no cap (process every selected character). Use this to try one
# character before committing hours of GPU time.
MAX_CHARACTERS_PER_LOCK = 0  #@param {type:"integer"}

# Run the scripts headless (no blocking Review UI inside each script).
# The Review UI is started ONCE by Cell 10 behind a tunnel.
RUN_HEADLESS = True  #@param {type:"boolean"}

START_TUNNEL = True  #@param {type:"boolean"}

# Cell 11: push exported/approved output back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# GitHub fine-grained/classic PAT (Settings -> Developer settings -> Tokens).
# Needs Contents: Read+Write on the repo.  Leave empty if the repo is public
# AND your push works without auth; Colab usually has no credential helper,
# so a PAT is required to push from here.
GITHUB_TOKEN = ""  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")

# Resolve the selected character list ('all' expands to the Universe seeds).
CHARACTER_LIST = [c.strip() for c in CHARACTERS.split(",") if c.strip()]


In [ ]:
#@title 2. Mount Google Drive (catalog.db only)

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive ready (holds catalog.db only):", DRIVE_ROOT)


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    # Full clone (not --depth 1) so Cell 11 can push approved output back.
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "timm", "diffusers", "transformers"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 3b. Preview the locking scope (no GPU spent)

# Runs AFTER Cell 3 (repo clone). Validates CHARACTER_LIST against the
# checked-out Universe and prints which locks will run per character, so a
# wrong character name is caught before it costs GPU hours.

import sys
sys.path.insert(0, REPO)

from src.universe.catalog import discover_characters

seeds = discover_characters(f"{REPO}/Universe")
if not seeds:
    raise SystemExit(f"No characters found under {REPO}/Universe - check the clone.")

if "all" in CHARACTER_LIST or "*" in CHARACTER_LIST:
    CHARACTER_LIST[:] = [s.name for s in seeds]

wanted = {c.lower() for c in CHARACTER_LIST}
matched = [s for s in seeds if s.name.lower() in wanted]
missing = sorted(wanted - {s.name.lower() for s in matched})
if missing:
    names = ", ".join(repr(s.name) for s in seeds)
    raise SystemExit(f"Unknown character(s): {missing}. Available: {names}")

if MAX_CHARACTERS_PER_LOCK and MAX_CHARACTERS_PER_LOCK > 0:
    matched = matched[:MAX_CHARACTERS_PER_LOCK]

LOCKS = [
    ("identity_lock", "references", "reference sheets (front / 3/4 / profile / back), 80 candidates/angle"),
    ("face_lock",      "expressions", "expression library (PromptBuilder expressions), 60 candidates each"),
    ("body_lock",      "poses",       "pose library (PromptBuilder poses), 60 candidates each"),
    ("wardrobe",       "outfits",     "wardrobe library (12+ outfits), candidates per outfit"),
]

print("=" * 72)
print(f"  CHARACTERS ({len(matched)}):")
for s in matched:
    print(f"    - {s.name}")
print("  LOCK STAGES:")
for script, _subdir, what in LOCKS:
    print(f"    - {script}: {what}")
print("=" * 72)
print("  Free-tier T4 note: ~2 min/image. The identity lock alone is")
print("  320 candidates per character (~10h on a T4). Prefer one character.")
print("  Set MAX_CHARACTERS_PER_LOCK = 1 in Cell 1 to trial-run safely.")


In [ ]:
#@title 4. Install ComfyUI

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])
print("ComfyUI ready at", COMFY)


In [ ]:
#@title 5. Download models (Colab disk, NOT Drive)

MODELS = {
    "colab-gpu": {
        "checkpoints/flux1-dev.safetensors":
            "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    },
}[BRANCH]

import shutil

# Default: Colab ephemeral disk (fits free tier, re-downloaded after reset).
# Opt-in: Drive cache, but only if your Drive actually has ~17.25 GB free.
cache_root = f"{DRIVE_ROOT}/models" if CACHE_MODELS_IN_DRIVE else f"{WORK}/models"

for rel, url in MODELS.items():
    cached = f"{cache_root}/{rel}"
    link = f"{COMFY}/models/{rel}"
    if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        # N-08: stop before a download exhausts the free tier's disk.
        free_gb = shutil.disk_usage("/content").free / 1e9
        if free_gb < 4:
            raise SystemExit(
                f"Only {free_gb:.1f} GB free on /content - cannot fit the "
                "~17.25 GB fp8 model. Stop other runtimes or free disk space."
            )
        print(f"Downloading {rel} ...")
        run(["wget", "-q", "-c", "-O", cached, url])
    # N-10: fp8 Flux is ~17.25 GB; a partial download is a red flag.
    size_gb = os.path.getsize(cached) / 1e9
    if size_gb < 17.25 * 0.9:
        raise SystemExit(
            f"Model looks truncated: {size_gb:.2f} GB cached - re-run this "
            "cell (wget -c resumes) or delete the file and restart."
        )
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.lexists(link) and not os.path.islink(link):
        os.remove(link)
    if not os.path.islink(link):
        try:
            os.symlink(cached, link)
        except OSError:
            shutil.copyfile(cached, link)
    print(f"OK {rel} ({size_gb:.2f} GB)")

print("Model cache:", cache_root)


In [ ]:
#@title 6. Start the ComfyUI server (auto-restart helper)

# Uses colab/comfy_helpers.py so the lock cells can restart the server
# after a Colab VM recycle instead of failing with Connection refused.
import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)
print("ComfyUI ready on :" + str(COMFYUI_PORT))


In [ ]:
#@title 7. Verify the GPU

import torch

assert torch.cuda.is_available(), "GPU runtime required - set Runtime > Change runtime type > T4 GPU."
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
#@title 8. Run the Progressive Locking Pipeline (4 lock stages)

# Runs the four lock scripts IN ORDER for each selected character.  Each
# script is called headless (--no-review-ui), so they run sequentially
# instead of each blocking on its own Review UI server.  The single Review
# UI is started by Cell 10.

# Per-character output subdirectory inside Universe/Characters/<name>/.
LOCK_DIRS = {
    "identity_lock": "references",
    "face_lock": "expressions",
    "body_lock": "poses",
    "wardrobe": "outfits",
}
LOCK_SCRIPTS = ["generate_identity_lock.py", "generate_face_lock.py",
                "generate_body_lock.py", "generate_wardrobe.py"]

import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

os.chdir(REPO)
processed = 0
for script in LOCK_SCRIPTS:
    subdir = LOCK_DIRS[script.replace("generate_", "").replace(".py", "")]
    print(f"\n{'=' * 72}")
    print(f"  STAGE: {script}  (outputs -> Universe/Characters/<name>/{subdir})")
    print(f"{'=' * 72}")
    for i, name in enumerate(list(CHARACTER_LIST), start=1):
        if MAX_CHARACTERS_PER_LOCK and i > MAX_CHARACTERS_PER_LOCK:
            print(f"  -- max {MAX_CHARACTERS_PER_LOCK} reached, stopping this stage --")
            break
        processed += 1
        universe_dir = f"{REPO}/Universe/Characters/{name}/{subdir}"
        print(f"\n--- [{i}/{len(CHARACTER_LIST)}] {script}: '{name}' ---")
        cmd = [sys.executable, f"scripts/{script}",
               "--comfyui-url", f"http://localhost:{COMFYUI_PORT}",
               "--character", name,
               "--universe-dir", universe_dir,
               "--db-path", DB]
        if RUN_HEADLESS:
            cmd.append("--no-review-ui")
        run(cmd)

print(f"\nProgressive Locking Pipeline complete ({processed} character-stage runs).")
print("Next: Cell 9 exports approved PNGs, Cell 10 opens the Review UI to approve.")


In [ ]:
#@title 9. Export real PNGs into the Colab file tree (not Drive)

os.chdir(REPO)
etype = "--asset-types character"  # characters only; locks produce char assets
!python scripts/export_assets.py --db {DB} --backend comfyui --comfyui-url http://localhost:{COMFYUI_PORT} --scope characters --size 1024 --universe {REPO}/Universe --world {REPO}/World --assets {REPO}/Assets


In [ ]:
#@title 10. Launch the Review UI and tunnel (approve the locks)

# Started ONCE after the pipeline so you can review and approve the
# shortlisted candidates per character.  Approve & Promote locks the
# "winner" asset that becomes the reference/expression/pose/outfit.

from src.review_ui.app import create_app
from src.universe.batch_generator import resolve_backend

app = create_app(
    db_path=DB,
    generation_backend=resolve_backend("comfyui", comfyui_url=f"http://localhost:{COMFYUI_PORT}"),
    universe_dir=f"{REPO}/Universe",
    world_dir=f"{REPO}/World",
    assets_dir=f"{REPO}/Assets",
    persist_generated_images=True,
)

import socket
import threading
import time
import uvicorn

# Skip rebinding when the UI from an earlier run is still listening.
ui_alive = False
probe = socket.socket()
probe.settimeout(2)
try:
    probe.connect(("127.0.0.1", UI_PORT))
    ui_alive = True
except Exception:
    ui_alive = False
finally:
    probe.close()

if ui_alive:
    print(f"Review UI already running on :{UI_PORT}")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=UI_PORT, log_level="warning")
    threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
    print(f"Review UI starting on :{UI_PORT} ...")

if START_TUNNEL:
    import re
    import shutil

    # localtunnel needs Node.js/npm, which Colab does not always ship.  Install
    # once, then use the cached `lt` binary directly (no npx re-fetch each run).
    if not shutil.which("lt"):
        run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"])
        run(["npm", "install", "-g", "--silent", "localtunnel"])
    lt = shutil.which("lt") or "lt"

    def open_tunnel(port, name):
        out = open(f"{WORK}/{name}.log", "w")
        return subprocess.Popen(
            [lt, "--port", str(port)],
            stdout=out, stderr=subprocess.STDOUT,
        )

    p1 = open_tunnel(COMFYUI_PORT, "tunnel_comfyui")
    p2 = open_tunnel(UI_PORT, "tunnel_ui")

    # Poll up to ~60s: the URL only appears once localtunnel connects to its
    # relay, and the first run also pays the npm download cost.
    urls = {}
    for _ in range(30):
        time.sleep(2)
        for name in ("tunnel_comfyui", "tunnel_ui"):
            txt = open(f"{WORK}/{name}.log").read()
            found = re.findall(r"https://[a-z0-9-]+\.loca\.lt", txt)
            if name not in urls or not urls[name]:
                urls[name] = found
        if urls.get("tunnel_comfyui") and urls.get("tunnel_ui"):
            break

    for name in ("tunnel_comfyui", "tunnel_ui"):
        found = urls.get(name) or []
        print(name, "->", found if found else "no URL yet")
        if not found:
            tail = open(f"{WORK}/{name}.log").read().splitlines()[-3:]
            print("   log tail:", tail)
    print()
    print("Review UI:", urls.get("tunnel_ui"))
    print("ComfyUI:  ", urls.get("tunnel_comfyui"))


In [ ]:
#@title 11. Training-readiness gate + sync approved output

# Approve/lock assets in the Review UI (Cell 10 tunnel), then run this cell.
# It prints the per-character approved-asset table that feeds
# `train_lora.py build-dataset` (>= 20 approved per character), then syncs.

import sqlite3

con = sqlite3.connect(DB)
rows = con.execute(
    "SELECT name, COUNT(*) FROM assets JOIN characters ON characters.id = assets.character_id "
    "WHERE assets.state = 'approved' GROUP BY characters.name ORDER BY characters.name"
).fetchall()
con.close()

print("=" * 72)
print("  APPROVED ASSETS PER CHARACTER (training-dataset readiness)")
print("=" * 72)
for name, cnt in rows:
    ok = "READY" if cnt >= 20 else f"{20 - cnt} more"
    print(f"    {name:28s} {cnt:4d}  {ok}")
if not rows:
    print("    (no approved assets yet - open the Review UI and approve)")
print("=" * 72)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import auto_sync

    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f"Progressive Locking output {datetime.now():%Y-%m-%d %H:%M}")
else:
    import zipfile
    from google.colab import files

    zip_path = f"{WORK}/identity_lock_export.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{REPO}/catalog.db", "catalog.db")
        for root, _dirs, names in os.walk(f"{REPO}/Assets"):
            for name in names:
                full = os.path.join(root, name)
                z.write(full, os.path.relpath(full, REPO))
    files.download(zip_path)
    print("Downloaded identity_lock_export.zip (catalog.db + Assets).")


## Next steps

- **Approve assets in the Review UI** (Cell 10 tunnel), then re-run Cell 11 to
  push the updated `catalog.db` + exported PNGs to GitHub, or download a zip.
- **More characters**: pick `CHARACTERS` in Cell 1 (run Cell 3b to preview the
  workload first), then re-run Cells 8-11. The pipeline skips nothing - each
  character-stage produces a fresh candidate pool.
- **LoRA training**: once a character has >= 20 approved assets, open
  `AnimationStudio_Colab_Training.ipynb` - its `build-dataset` step consumes
  the approved set this notebook produces.
- After a VM reset: models re-download (Cell 5), `catalog.db` comes back from
  Drive, and previously approved assets are pulled from GitHub.

## Troubleshooting

- CUDA out of memory: shrink the scope in Cell 1 (fewer characters), or switch
  to the L4/A100 runtime.
- Model download stalls: re-run Cell 5 (`wget -c` resumes into the cache).
- ComfyUI failed to start: read the log tail printed by Cell 6.
- Drive full: only `catalog.db` should be on Drive. If an older run cached
  `models/` under `DRIVE_ROOT`, delete it to reclaim ~17.25 GB.
- `git push` fails: Cell 11 pushes with the `REPO_URL` credentials - for
  HTTPS use a personal access token (not your password) when prompted.
